In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from embedders import EmbedderModel, EmbedderModelConfig
from ChromaConnector import (ChromaConnection, 
                             VectorDBConnectionConfig, 
                             VectorDBInstance)

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
data_path = "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/qa_dataset.csv"
data = pd.read_csv(data_path)
data.head()

,question,answer,relevant_context_id,metadata
0,When did Beyonce start becoming popular?,in the late 1990s,0,{'base_id': '56be85543aeaaa14008c9063'}
1,What areas did Beyonce compete in when she was...,singing and dancing,0,{'base_id': '56be85543aeaaa14008c9065'}
2,When did Beyonce leave Destiny's Child and bec...,2003,0,{'base_id': '56be85543aeaaa14008c9066'}
3,In what city and state did Beyonce grow up?,"Houston, Texas",0,{'base_id': '56bf6b0f3aeaaa14008c9601'}
4,In which decade did Beyonce become famous?,late 1990s,0,{'base_id': '56bf6b0f3aeaaa14008c9602'}


In [5]:
data.shape

(86821, 4)

In [4]:
%%time
emb_model = EmbedderModel(EmbedderModelConfig())
question_embs = emb_model.encode_queries(data.question.to_list())

CPU times: user 43.5 s, sys: 40.7 s, total: 1min 24s
Wall time: 31 s


In [6]:
len(question_embs)

86821

In [7]:
ids = ["id" + str(i) for i in range(len(question_embs))]

In [8]:
db_save_path = "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/chroma/dbs/v3"
db_info = {'db': 'squadv2', 'table': 'questions'}
vdb_conf = VectorDBConnectionConfig(path=db_save_path, db_info=db_info)
connector = ChromaConnection(config=vdb_conf)

In [9]:
connector.count_items()

0

In [10]:
new_data = [
    VectorDBInstance(id=i,
                     document=doc,
                     embedding=emb
                    )
    for i, doc, emb in zip(ids, data.question.to_list(), question_embs)]

In [14]:
batch_size = 10000
for i in range(0, len(new_data), batch_size):
    connector.create(new_data[i:i + batch_size])

In [15]:
connector.count_items()

86821

In [16]:
connector.read(['id80000'])

[VectorDBInstance(id='id80000', document='What did Renaissance painters call the pigment made from cochineal?', embedding=[0.05860181525349617, -0.05813271924853325, -0.06837169826030731, -0.0898294523358345, 0.05095094069838524, -0.09356451034545898, 0.04161927103996277, 0.018139339983463287, 0.04932406172156334, 0.01100807636976242, 0.01627397909760475, 0.0038197129033505917, 0.05844626948237419, -0.04188612475991249, -0.03400953859090805, 0.03120654635131359, 0.07933180034160614, 0.019503632560372353, -0.0793970376253128, -0.11052089184522629, 0.03816618025302887, -0.03379843756556511, -0.03000185266137123, 0.059778157621622086, 0.042053669691085815, 0.027821741998195648, -0.01202701497823, -0.02829272672533989, 0.04418540745973587, -0.043793950229883194, -0.06243617460131645, -0.06293812394142151, 0.03952658548951149, -0.020506253466010094, -0.0006001214496791363, 0.02721378207206726, -0.07507413625717163, -0.07050999999046326, 0.033585067838430405, -0.024157484993338585, -0.033319